# Predicting charity failure: what the literature already does

This notebook is the conceptual entry point to the project. It sets out the four
studies we benchmark against, the measures each proposed, and the choices we had
to make to bring them to Canadian T3010 data. Full references, and the two
forecasting/hazard-model papers cited elsewhere in the series, are on the
[References](../references.md) page (BibTeX in [`refs.bib`](refs.bib)); PDFs
are in `reading/` in a full checkout of this repository (not published here,
as the papers themselves are under journal copyright).

The rest of the series:

| Notebook | What it covers |
|---|---|
| `01-data-and-outcomes` | The T3010 extract, the charity-year panel, and what "failure" means here |
| `02-features-and-benchmarks` | The Tuckman-Chang ratios and the two logit replications |
| `03-model-results` | Out-of-time benchmark comparison, a deep network, a Bayesian alternative, and a watchlist |
| `04-schedule-6-deep-dive` | Section D filings excluded; what the detailed return's extra fields buy |
| `05-implementation-and-extensions` | Why this is useful, two ways to deploy it, and what better data would buy |

## Tuckman and Chang (1991)

*A Methodology for Measuring the Financial Vulnerability of Charitable Nonprofit
Organizations.* The founding paper. {cite:t}`tuckman1991` define a nonprofit as
financially vulnerable if it is "likely to cut back its service offerings
immediately when it experiences a financial shock", and propose four accounting
measures to detect that condition **before** the shock arrives:

1. **Equity balance** — net assets over total revenue. Thin equity means no
   buffer.
2. **Revenue concentration** — a Herfindahl index over revenue sources. A
   charity funded by one grant is one grant away from trouble.
3. **Administrative cost ratio** — administration over total expenses. This one
   runs *backwards* from intuition: administrative spending is discretionary
   slack that can be cut in a crisis, so a charity with almost none has nothing
   left to give up.
4. **Operating margin** — surplus over revenue. Persistent deficits eat the
   buffer.

The method is not a fitted model. Each measure is compared with the sector
distribution; a charity in the risky quintile is flagged; the flags are counted.
Zero is healthy, four is "severely at risk". That design is a strength — it
needs no historical failures to calibrate — and a weakness, because a 0-4 count
can only sort organisations into five groups.

## Greenlee and Trussel (2000)

*Predicting the Financial Vulnerability of Charitable Organizations.*
{cite:t}`greenlee2000` keep the four Tuckman-Chang ratios but stop counting
flags and start estimating. They put the ratios into a logistic regression,
and they need an observable outcome to regress on, so they define one:

> a nonprofit is **vulnerable** if its net assets fall by 20% or more over three
> years.

This is the definition the whole subsequent literature inherits, and it is worth
being clear about what it is and is not. It is a measure of *financial distress*.
It is not failure: an organisation can lose a quarter of its endowment and
continue for decades, and an organisation can dissolve with its balance sheet
intact.

## Trussel (2002)

*Revisiting the Prediction of Financial Vulnerability.* {cite:t}`trussel2002`
re-specifies the model on 94,000 organisations and lands on a different four:

* revenue concentration (kept from Tuckman-Chang),
* surplus margin (the operating margin, renamed),
* **debt ratio** — total liabilities over total assets (new),
* administrative cost ratio (kept, though it is the weakest of the four),

plus a **size** control and **sector** fixed effects. The finding that size
matters this much is the one we most wanted to test in Canada, because it raises
an awkward question the paper does not press: how much of the model's
performance is the four ratios, and how much is simply knowing that small
organisations fail more often? We answer it directly with a size-only baseline
in `03-model-results`.

## Burde (2016)

*Prediction of Financial Vulnerability to Funding Instability.*
{cite:t}`burde2016` argues the literature has conflated two different things —
vulnerability to a *funding shock* and vulnerability to *insolvency* — and
that the Tuckman-Chang measures speak mostly to the first. We take the point
seriously enough to model both a distress outcome and an exit outcome
separately, rather than assuming one stands in for the other.

## Alam, Gao and Jones (2021)

*Corporate failure prediction: An evaluation of deep learning vs discrete hazard
models.* The one modern machine-learning benchmark in this set, and the only one
not drawn from the nonprofit literature. {cite:t}`alam2021` pit a deep model
against a **discrete-time hazard model** — the standard panel-data workhorse in
the corporate failure literature, after {cite:t}`shumway2001` and
{cite:t}`hillegeist2004` — on 641,667 firm-months of North American listed
companies, 2001-2018.

Their deep model is not an off-the-shelf network. Its contribution is a way to
feed *variable-length panel histories* into a classifier: each firm's history is
a D x t matrix, and no two firms have the same t. They take the SVD of that
matrix and keep the first d left singular vectors, which represents the firm as
a point on a **Grassmann manifold** — a fixed-size object regardless of how long
the firm has existed. A GrNet then learns over those points with Riemannian
gradient descent.

Two things to keep in view when reading their result.

The headline "93.71% versus 86.95%" is **sensitivity**, not accuracy. On overall
accuracy and specificity the discrete hazard model wins. The deep model trades
false alarms for caught failures — a defensible trade when a missed failure is
far costlier, but a threshold choice rather than a uniform improvement.

Their failure rate is 21.5%. Ours is 2.0%. Percentages of correctly classified
cases behave very differently at those two base rates, which is why
`03-model-results` reports ranking metrics instead.

## What changes when you move to Canada

The T3010 is not the Form 990, and four differences shaped the design.

**A registry, not a survey.** Every registered Canadian charity must file a T3010
annually for as long as it is registered. Disappearance from the filing record is
therefore informative in a way that absence from a US sample is not: it means the
registration ended. That gives us an outcome much closer to *corporate failure*
than the 20%-net-asset-decline proxy, and it is our primary target.

**Two filing tiers in one file.** About a third of Canadian charity-years are
filed on the short Section D form rather than the detailed Schedule 6. Section D
filers are never asked for cash, payables, investment income, compensation or
fundraising expense. A blank in those lines is *not* a zero, and treating it as
one would silently turn the filing tier into a financial ratio. The handling is
in `charity_risk.fields` and is the single most consequential data decision in
the project.

**A coarser revenue breakdown.** The Tuckman-Chang concentration index needs
revenue by source. Section D and Schedule 6 disaggregate differently, so we
harmonise both onto eight common buckets before computing the index — otherwise
the measure would be systematically higher for short-form filers purely because
they report fewer lines.

**A five-year window.** The extract runs 2019-2023. That bounds what can be
asked: a three-year vulnerability horizon can only be resolved for two cohorts,
and a permanent-exit label needs enough lookahead years to distinguish a
deregistration from a late filing. `01-data-and-outcomes` works through the
consequences.